# HW1 Q1 · 2-head word 빈도 (5팀)

**문제**: `sherlock.txt`의 단어 중 길이가 2보다 큰 단어만 골라, 소문자로 바꾼 앞 두 글자(2-head word)별 빈도를 센다. 빈도가 **32 미만**인 2-head word들의 빈도를 모두 더해 출력한다.

**팀 결정**
- 단어 = 공백(`' '`) 기준으로 자른 토큰. 구두점은 그대로 둔다 (`'I.'`도 한 단어).
- 길이 조건은 문제 문장대로 `len(w) > 2`.
  예시의 `IN`, `my`는 원본 파일에서 앞뒤에 다른 문자가 없는 길이 2 단어임을 확인함
  (특수문자·공백문자 가설 검토 후 기각). 예시 목록은 `len(w) >= 2`를 적용한 결과와 정확히 일치하므로
  (길이 1인 `I`만 빠짐), 예시가 다른 기준으로 작성된 것으로 보고 문제 문장을 따름.
  (`>= 2`로 하면 결과 830)

**결과**: `772`

**흐름** (원소 모양)
`줄` → flatMap → `단어` → filter → `길이 3 이상 단어` → map → `(head, 1)` → reduceByKey → `(head, 빈도)` → filter → `(head, 빈도<32)` → map → `빈도` → reduce → `772`


In [1]:
# HW1 Q1 - 2-head word 빈도
# 환경 준비: PySpark 설치, SparkContext 생성
!pip -q install pyspark
from pyspark import SparkContext
sc = SparkContext.getOrCreate()

In [2]:
# 데이터 내려받기 (textFile은 https 주소를 직접 못 읽으므로 파일로 받음, 내용은 손대지 않음)
!wget -q -O sherlock.txt https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/quiz/sherlock.txt
!head -3 sherlock.txt

A STUDY IN SCARLET.




In [3]:
# 0. 읽기: 원소 = 파일의 줄 하나(문자열). 빈 줄도 '' 원소 하나
rdd = sc.textFile("sherlock.txt")

# 1. 줄 -> 단어
#    lambda는 줄 하나를 받아 단어 리스트 하나를 돌려줌: 'A STUDY IN' -> ['A', 'STUDY', 'IN']
#    flatMap이 그 리스트를 펼쳐서 단어 하나하나를 원소로 만듦 (map이면 원소가 줄마다 리스트로 남아
#    다음 filter의 len이 단어 길이가 아니라 '줄의 단어 수'를 세게 됨)
words = rdd.flatMap(lambda line: line.split(' '))

# 2. 길이 > 2인 단어만 남김. 'STUDY' -> 남음 / 'A', 'IN', 'I.', ''(공백 두 칸 사이) -> 빠짐
#    반드시 앞 두 글자로 자르기 전에 해야 함. 자르고 나면 모든 head가 길이 2 이하라 원래 길이를 알 수 없음
long_words = words.filter(lambda w: len(w) > 2)

# 3. 단어 -> (head, 1)
#    lower()를 먼저 해서 'The'와 'the'가 같은 head 'th'로 묶이게 함
#    [:2]는 0번, 1번 글자 (끝 번호 2는 포함 안 됨): 'study' -> 'st'
#    원소 하나 -> 원소 하나라서 map
pairs = long_words.map(lambda w: (w.lower()[:2], 1))

# 4. head별 빈도: ('st', 1), ('st', 1), ... -> ('st', 빈도)
#    lambda의 x, y는 튜플이 아니라 같은 키의 '값' 두 개(1, 1). 키는 reduceByKey가 들고 있음
#    셔플이 일어나는 곳은 여기 한 곳(wide). 셔플 전에 각 파티션 안에서 같은 키끼리 먼저 더해 두므로
#    ('st', 1) 여러 개 대신 ('st', 부분합) 하나만 네트워크로 보냄 -> groupByKey보다 이동량이 적음
counts = pairs.reduceByKey(lambda x, y: x + y)

# 5. 빈도가 32 미만(< 32, 32는 제외)인 것만 남김
#    filter의 lambda는 원소 (head, 빈도) 튜플을 통째로 받으므로 kv[1]이 빈도
small = counts.filter(lambda kv: kv[1] < 32)

# 6. 빈도만 꺼내(map: (head, 빈도) -> 빈도) 모두 더함(reduce)
#    reduce의 lambda 모양은 (숫자, 숫자) -> 숫자. 덧셈은 결합법칙이 성립해서 파티션별로 나눠 더해도 결과가 같음
#    reduce는 action -> 여기서 처음으로 0~6단계 계획 전체가 실행됨 (그 전 줄들은 계획만 세움, lazy)
total = small.map(lambda kv: kv[1]).reduce(lambda a, b: a + b)

In [4]:
print(total)

772
